# Abacus Backlight Paste Scaling Diagnostics

This notebook reads the JSON products written by `benchmark-pixel-work` and `paste-split` timing instrumentation. It is intended for quick ablation studies of CPU HEALPix neighbor generation, JAX/GPU map generation, split balance, and extrapolated full-sky runtime.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path('/mnt/ceph/users/spandey/ltu-godmax/GODMAX')
root = repo / 'data/xDESI/processed/abacus_backlight'
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

In [ ]:
bench_rows = []
for path in sorted(root.glob('stage31_pz*/measurements/pixel_work_benchmark*.json')):
    payload = json.loads(path.read_text())
    run = path.parts[-3]
    for row in payload.get('rows', []):
        bench_rows.append({'run': run, 'path': str(path), **row})
bench = pd.DataFrame(bench_rows)
bench

In [ ]:
if len(bench):
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for (nside, shortcut), group in bench.groupby(['nside', 'single_pixel_angle_factor']):
        g = group.sort_values('workers')
        label = f'Nside {nside}, shortcut={shortcut:g}'
        ax.plot(g['workers'], g['halos_per_s'], marker='o', linewidth=2, label=label)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('CPU workers')
    ax.set_ylabel('Pixel-neighbor throughput [halos/s]')
    ax.set_title('HEALPix neighbor generation scaling')
    ax.legend(frameon=False)
    fig.tight_layout()
else:
    print('No benchmark JSON files found yet.')

In [ ]:
if len(bench) and 'compare_exact_pixel_mismatch_fraction' in bench.columns:
    comp = bench.dropna(subset=['compare_exact_pixel_mismatch_fraction']).copy()
    if len(comp):
        fig, ax = plt.subplots(figsize=(7, 4.5))
        sc = ax.scatter(comp['halos_per_s'], comp['compare_exact_pixel_mismatch_fraction'],
                        c=comp['single_pixel_angle_factor'], s=70, cmap='viridis')
        ax.set_xlabel('Pixel-neighbor throughput [halos/s]')
        ax.set_ylabel('Pixel mismatch fraction vs exact query_disc')
        ax.set_title('Single-pixel shortcut speed/accuracy tradeoff')
        fig.colorbar(sc, ax=ax, label='shortcut angle factor')
        fig.tight_layout()
    else:
        print('No compare-exact benchmark rows found yet.')
else:
    print('Run benchmark-pixel-work with --compare-exact to populate shortcut accuracy rows.')

In [ ]:
timing_rows = []
for path in sorted(root.glob('stage31_pz*/maps/**/*.timing.json')):
    payload = json.loads(path.read_text())
    run = path.parts[-4]
    for chunk in payload.get('chunks', []):
        row = {'run': run, 'path': str(path)}
        for key in ['nside', 'split_index', 'num_splits', 'pixel_workers', 'pixel_pool_chunksize', 'single_pixel_angle_factor']:
            row[key] = payload.get(key)
        row.update(chunk)
        row['gpu_wl_extra_total_time_s'] = sum((chunk.get('gpu_wl_extra_time_s') or {}).values())
        timing_rows.append(row)
timing = pd.DataFrame(timing_rows)
timing

In [ ]:
if len(timing):
    cols = ['pixel_time_s', 'transfer_time_s', 'gpu_main_time_s', 'gpu_wl_extra_total_time_s', 'gpu_cmb_time_s']
    by_run = timing.groupby(['run', 'nside'])[cols].sum(min_count=1).fillna(0.0)
    ax = by_run.plot(kind='bar', stacked=True, figsize=(9, 5), colormap='tab20')
    ax.set_ylabel('Recorded split runtime contribution [s]')
    ax.set_title('Paste split bottleneck breakdown')
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
else:
    print('No paste-split timing JSON files found yet. Future paste-split runs will write them next to the HDF5 split products.')

In [ ]:
if len(timing):
    summary = timing.groupby(['run', 'nside']).agg(
        halos=('n_halos', 'sum'),
        pairs=('n_pairs', 'sum'),
        pixel_s=('pixel_time_s', 'sum'),
        gpu_main_s=('gpu_main_time_s', 'sum'),
        gpu_wl_s=('gpu_wl_extra_total_time_s', 'sum'),
        gpu_cmb_s=('gpu_cmb_time_s', 'sum'),
    )
    summary['pairs_per_halo'] = summary['pairs'] / summary['halos']
    summary['pixel_halos_per_s'] = summary['halos'] / summary['pixel_s'].replace(0, np.nan)
    summary['gpu_pairs_per_s'] = summary['pairs'] / (summary['gpu_main_s'] + summary['gpu_wl_s'] + summary['gpu_cmb_s']).replace(0, np.nan)
    display(summary)
else:
    print('Run paste-split with the updated code to populate split timing summaries.')

In [ ]:
gpu_rows = []
for path in sorted(root.glob('stage31_pz*/measurements/gpu_chunk_benchmark*.json')):
    payload = json.loads(path.read_text())
    run = path.parts[-3]
    for row in payload.get('rows', []):
        gpu_rows.append({
            'run': run,
            'path': str(path),
            'nside': payload.get('nside'),
            'n_halos': payload.get('n_halos'),
            'n_pairs': payload.get('n_pairs'),
            'pixel_time_s': payload.get('pixel_time_s'),
            **row,
        })
gpu = pd.DataFrame(gpu_rows)
gpu

In [ ]:
if len(gpu):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for fused, group in gpu.groupby('fused'):
        label = 'Fused profile passes' if fused else 'Separate profile passes'
        ax.scatter(group['n_pairs'], group['runtime_s'], s=70, label=label)
    ax.set_xlabel('Halo-pixel pairs')
    ax.set_ylabel('Map-generation runtime [s]')
    ax.set_title('JAX profile evaluation ablation')
    ax.legend(frameon=False)
    fig.tight_layout()
else:
    print('No GPU chunk benchmark JSON files found yet.')